# flip.simulation — Forward Model Demo

This notebook walks through the `flip.simulation` submodule end-to-end:

1. Setting up a cosmology and a `ForwardModel`
2. Generating density and velocity fields
3. Extracting line-of-sight velocities at galaxy positions
4. Evaluating a field-level Gaussian likelihood
5. Computing gradients through the full pipeline
6. (Optional) Running a gradient-based fit with `SimulationFitter`

**Required packages:** `jaxpm`, `jax_cosmo`, and optionally `jaxopt` for the fitter.

```bash
pip install jaxpm jax-cosmo jaxopt
```

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# flip simulation submodule
from flip.simulation.generator import ForwardModel, get_cosmology
from flip.simulation.painter import (
    sky_to_cartesian,
    cartesian_to_box_frame,
    compute_los_velocity,
    interpolate_velocity_to_positions,
)
from flip.simulation.likelihood import VelocityFieldLikelihood, log_likelihood_gaussian

print('JAX version:', jax.__version__)
print('devices:', jax.devices())

## 1. Cosmology and ForwardModel setup

In [ ]:
# Fiducial cosmology
cosmo = get_cosmology(omega_m=0.3, sigma8=0.8, h=0.6774)
print(cosmo)

# Simulation box: 256 Mpc/h per side, 64^3 mesh → 4 Mpc/h cells
# (use 32^3 for fast testing)
MESH = (32, 32, 32)
BOX  = (256., 256., 256.)

model = ForwardModel(
    mesh_shape=MESH,
    box_size=BOX,
    a_final=1.0,   # z = 0
    lpt_only=True, # Zel'dovich approximation — fast and differentiable
)
print(model)

## 2. Generate density and velocity fields

In [ ]:
seed = jax.random.PRNGKey(0)

density_field, velocity_field = model.get_fields(cosmo, seed=seed)

print(f'density_field shape: {density_field.shape}')
print(f'velocity_field shape: {velocity_field.shape}')
print(f'δ mean = {float(jnp.mean(density_field)):.4f}  (should be ≈ 0)')
print(f'δ std  = {float(jnp.std(density_field)):.4f}')
print(f'v rms  = {float(jnp.sqrt(jnp.mean(velocity_field**2))):.1f} km/s')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Density slice
ax = axes[0]
im = ax.imshow(np.array(density_field[:, :, MESH[2]//2]), origin='lower',
               cmap='RdBu_r', vmin=-2, vmax=2)
plt.colorbar(im, ax=ax, label=r'$\delta$')
ax.set_title('Density contrast δ (z-slice)')

# Velocity slices (x and y components)
for i, (ax, label) in enumerate(zip(axes[1:], [r'$v_x$ [km/s]', r'$v_y$ [km/s]'])):
    vmax = float(jnp.std(velocity_field[..., i])) * 2
    im = ax.imshow(np.array(velocity_field[:, :, MESH[2]//2, i]),
                   origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    plt.colorbar(im, ax=ax, label=label)
    ax.set_title(f'Velocity {label} (z-slice)')

plt.tight_layout()
plt.show()

## 3. Extract LOS velocities at galaxy positions

We create a mock galaxy catalog with RA/Dec/r_com and extract the simulated
line-of-sight peculiar velocity at each galaxy.

In [ ]:
rng = np.random.default_rng(42)
N_gal = 200

# Mock galaxy catalog: RA/Dec in radians, r_com in Mpc/h
ra   = jnp.array(rng.uniform(0,   2*np.pi, N_gal))  # radians
dec  = jnp.array(rng.uniform(-0.3, 0.3,   N_gal))   # radians (small survey)
rcom = jnp.array(rng.uniform(50,   110.,  N_gal))    # Mpc/h

# Convert to Cartesian (observer at origin)
xyz_observer = sky_to_cartesian(ra, dec, rcom)

# Shift to box frame (observer at box centre)
xyz_box = cartesian_to_box_frame(xyz_observer, BOX)

print(f'Galaxy positions range (box frame):')
for i, ax in enumerate('xyz'):
    print(f'  {ax}: [{float(xyz_box[:,i].min()):.1f}, {float(xyz_box[:,i].max()):.1f}] Mpc/h')

In [ ]:
# Interpolate 3D velocity to galaxy positions
v3d_gal = interpolate_velocity_to_positions(velocity_field, xyz_box, BOX, MESH)

# Project onto LOS (r̂ from observer-centred positions)
v_los = compute_los_velocity(v3d_gal, xyz_observer)

print(f'v_los: mean={float(jnp.mean(v_los)):.1f}, std={float(jnp.std(v_los)):.1f} km/s')

plt.figure(figsize=(7, 4))
plt.hist(np.array(v_los), bins=30, edgecolor='k', color='steelblue', alpha=0.8)
plt.xlabel('$v_{\\rm los}$ [km/s]')
plt.ylabel('Count')
plt.title('Simulated LOS peculiar velocities')
plt.tight_layout()
plt.show()

## 4. Gaussian field-level likelihood

In [ ]:
# Observed velocities: use simulated + noise
sigma_obs = 150.0  # km/s measurement error
v_obs = np.array(v_los) + rng.normal(0, sigma_obs, N_gal)
var_obs = jnp.ones(N_gal) * sigma_obs**2

ll = log_likelihood_gaussian(v_los, jnp.array(v_obs), var_obs)
print(f'log L = {float(ll):.2f}')

# Likelihood is higher at the truth than at shifted velocity
ll_shifted = log_likelihood_gaussian(v_los + 200.0, jnp.array(v_obs), var_obs)
print(f'log L (200 km/s offset) = {float(ll_shifted):.2f}  (should be lower)')

## 5. Gradients through the full pipeline

The entire chain — from cosmological parameters through LPT simulation to the
likelihood — is differentiable via JAX.

In [ ]:
class MockDataVector:
    """Minimal stub: returns the observed velocities and variances."""
    def __init__(self, v_obs, var_obs):
        self._v = v_obs
        self._var = var_obs
    def give_data_and_variance(self, params=None):
        return self._v, np.array(self._var)

lik = VelocityFieldLikelihood(
    forward_model=model,
    data_vector=MockDataVector(v_obs, var_obs),
    positions_cartesian=xyz_box,
    seed=seed,
)

# Evaluate at fiducial cosmology
params_fid = {"omega_m": 0.3, "sigma8": 0.8}
neg_ll = lik(params_fid)
print(f'-log L (fiducial) = {float(neg_ll):.2f}')

# Gradient w.r.t. sigma8
def loss_sigma8(s8):
    return lik({"omega_m": 0.3, "sigma8": s8})

g_s8 = jax.grad(loss_sigma8)(0.8)
print(f'd(-log L)/d(sigma8) = {float(g_s8):.4f}')

# Gradient w.r.t. omega_m
def loss_omm(omm):
    return lik({"omega_m": omm, "sigma8": 0.8})

g_omm = jax.grad(loss_omm)(0.3)
print(f'd(-log L)/d(omega_m) = {float(g_omm):.4f}')

## 6. Likelihood as a function of σ₈

Scan the likelihood over σ₈ to check it peaks near the truth.

In [ ]:
sigma8_grid = np.linspace(0.5, 1.2, 15)
neg_ll_grid = [float(lik({"omega_m": 0.3, "sigma8": s8})) for s8 in sigma8_grid]

plt.figure(figsize=(7, 4))
plt.plot(sigma8_grid, neg_ll_grid, 'o-', color='steelblue')
plt.axvline(0.8, color='red', ls='--', label='truth σ₈=0.8')
plt.xlabel(r'$\sigma_8$')
plt.ylabel(r'$-\log\mathcal{L}$')
plt.title('Likelihood scan over σ₈')
plt.legend()
plt.tight_layout()
plt.show()

## 7. (Optional) Gradient-based optimization with SimulationFitter

Requires `jaxopt`. Comment out or skip if not installed.

In [ ]:
try:
    from flip.simulation.fitter import SimulationFitter

    fitter = SimulationFitter(
        likelihood=lik,
        initial_params={"omega_m": 0.25, "sigma8": 0.70},
        bounds=(
            {"omega_m": 0.1, "sigma8": 0.3},
            {"omega_m": 0.9, "sigma8": 1.5},
        ),
        solver="LBFGSB",
        maxiter=50,
    )

    best = fitter.run()
    print('Best-fit parameters:')
    for k, v in best.items():
        print(f'  {k} = {v:.4f}')

except ImportError:
    print('jaxopt not installed — skipping SimulationFitter demo.')

## 8. ForwardModel.from_survey_geometry

Convenience constructor that sizes the box to a given survey.

In [ ]:
# Typical PV survey: r_max ~ 300 Mpc/h, cell size 5 Mpc/h
model_survey = ForwardModel.from_survey_geometry(
    rcom_max=300.0,
    cell_size_mpc=5.0,
    z_survey=0.05,
    lpt_only=True,
)
print(model_survey)
print(f'N_part = {model_survey.mesh_shape[0]**3:,}')